<a href="https://colab.research.google.com/github/Ragul-S-2025/ML/blob/main/Feautre_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import Libraries and Load datasets

import pandas as pd

df = pd.read_excel('/content/Data_Train.xlsx')


In [2]:
df.shape

(10683, 11)

In [3]:
df.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302


In [4]:
df.isnull().sum()

,0
Airline,0
Date_of_Journey,0
Source,0
Destination,0
Route,1
Dep_Time,0
Arrival_Time,0
Duration,0
Total_Stops,1
Additional_Info,0


In [5]:
# Fill missing values in 'Total_Stops' with the mode and 'Route' with 'Unknown'
df['Total_Stops'] =df['Total_Stops'].fillna(df['Total_Stops'].mode()[0])
df['Route'] = df['Route'].fillna('Unknown')

In [6]:
df.isnull().sum()

,0
Airline,0
Date_of_Journey,0
Source,0
Destination,0
Route,0
Dep_Time,0
Arrival_Time,0
Duration,0
Total_Stops,0
Additional_Info,0


In [7]:
df.columns

Index(['Airline', 'Date_of_Journey', 'Source', 'Destination', 'Route',
       'Dep_Time', 'Arrival_Time', 'Duration', 'Total_Stops',
       'Additional_Info', 'Price'],
      dtype='object')

In [8]:

X=df[['Airline','Duration','Source', 'Destination','Duration']]

In [9]:
y=df[['Price']]

**Feature** Selection

 Recursive Feature Elimination

In [10]:
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler
def split_scalar(indep_X,dep_Y):
        X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)


        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test = sc.transform(X_test)
        return X_train, X_test, y_train, y_test

In [11]:
def classification_f1(classifier, X_test, y_test):
    y_pred = classifier.predict(X_test)
    from sklearn.metrics import f1_score
    f1 = f1_score(y_test, y_pred, average='weighted')
    return f1

In [12]:
def Linear(X_train, y_train, X_test, y_test):
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    classifier = LogisticRegression()
    classifier.fit(X_train, y_train)
    metrics = classification_f1(classifier, X_test, y_test)
    return metrics



In [13]:
def Decision(X_train, y_train, X_test, y_test):
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.metrics import f1_score
    classifier = DecisionTreeClassifier()
    classifier.fit(X_train, y_train)
    f1 = classification_f1(classifier, X_test, y_test)
    return f1

In [14]:
def random(X_train, y_train, X_test,y_test):
        from sklearn.ensemble import RandomForestClassifier
        classifier = RandomForestClassifier(n_estimators = 10, random_state = 0)
        classifier.fit(X_train, y_train)
        accuracy = classification_f1((classifier, X_test, y_test))
        return accuracy

In [15]:
def xgboost(X_train, y_train, X_test):
        from xgboost import XGBClassifier
        classifier = XGBClassifier(n_jobs=5, learning_rate=0.1, max_depth=10, random_state=1)
        classifier.fit(X_train, y_train)
        accuracy = classification_f1((classifier, X_test, y_test))
        return accuracy

In [16]:
"""from sklearn.feature_selection import RFE
def rfeFeature(indep_X, dep_Y, n):
    rfelist = []
    colnames_list = []
    # List to store column names for each model
    r2_values = []
    # List to store R2 values for each model

    from sklearn.linear_model import Linearclassfier
    lin = Linearclassfier()

    from sklearn.tree import DecisionTreeClassifier
    dec = DecisionTreeClassifier(random_state=0)

    from sklearn.ensemble import RandomForestClassifier
    rf = RandomForestClassifier(n_estimators=10, random_state=0)

    from xgboost import XGBClassifier
    xgb = XGBClassifier(n_jobs=5, learning_rate=0.1, max_depth=10, random_state=1)

    rfemodellist = [lin, dec, rf, xgb]

    for model in rfemodellist:
        log_rfe = RFE(estimator=model, n_features_to_select=n)
        log_fit = log_rfe.fit(indep_X, dep_Y)
        log_rfe_feature = log_fit.transform(indep_X)
        rfelist.append(log_rfe_feature)

        # Get the column names selected by RFE
        selected_columns = [col for col, selected in zip(indep_X.columns, log_rfe.support_) if selected]
        colnames_list.append(selected_columns)

        from sklearn.metrics import accuracy_score, f1_score

        # Fit the model and calculate and store the classification score
        X_train, X_test, y_train, y_test = split_scalar(pd.DataFrame(log_rfe_feature), dep_Y)
        model.fit(X_train, y_train)  # Fit the model

        # Predictions
        y_pred = model.predict(X_test)

        f1 = f1_score(y_test, y_pred, average="weighted")  # handles multi-class# Store results
        f1.append(f1)

        return rfelist, colnames_list, f1_score

# Return values: rfelist, colnames_list, scores_list
rfelist, colnames_list, scores_list = rfeFeature(X,y, 5)

# Print the selected column names and classification scores for each model
for model_name, selected_columns, scores in zip(["Logistic", "Decision", "Random", "XGBoost"], colnames_list, scores_list):
    acc, f1 = scores
    print(f"Model: {model_name}")
    print("Selected Columns:", selected_columns)
    print(f"Accuracy: {acc:.3f}, F1 Score: {f1:.3f}\n")
"""

'from sklearn.feature_selection import RFE\ndef rfeFeature(indep_X, dep_Y, n):\n    rfelist = []\n    colnames_list = []\n    # List to store column names for each model\n    r2_values = []\n    # List to store R2 values for each model\n\n    from sklearn.linear_model import Linearclassfier\n    lin = Linearclassfier()\n\n    from sklearn.tree import DecisionTreeClassifier\n    dec = DecisionTreeClassifier(random_state=0)\n\n    from sklearn.ensemble import RandomForestClassifier\n    rf = RandomForestClassifier(n_estimators=10, random_state=0)\n\n    from xgboost import XGBClassifier\n    xgb = XGBClassifier(n_jobs=5, learning_rate=0.1, max_depth=10, random_state=1)\n\n    rfemodellist = [lin, dec, rf, xgb]\n\n    for model in rfemodellist:\n        log_rfe = RFE(estimator=model, n_features_to_select=n)\n        log_fit = log_rfe.fit(indep_X, dep_Y)\n        log_rfe_feature = log_fit.transform(indep_X)\n        rfelist.append(log_rfe_feature)\n\n        # Get the column names selected

In [17]:
"""from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import pandas as pd

def rfeFeature(indep_X, dep_Y, n):
    rfelist = []
    colnames_list = []
    scores_list = []  # will store (accuracy, f1) for each model

    # Convert categorical features to numerical using one-hot encoding
    indep_X_encoded = pd.get_dummies(indep_X, columns=['Airline', 'Source', 'Destination'])

    # Convert 'Duration' to numerical (assuming format 'Xh Ym')
    def parse_duration(duration_str):
        if isinstance(duration_str, str):
            parts = duration_str.split(' ')
            hours = 0
            minutes = 0
            for part in parts:
                if 'h' in part:
                    hours = int(part.replace('h', ''))
                elif 'm' in part:
                    minutes = int(part.replace('m', ''))
            return hours * 60 + minutes
        return 0  # Handle NaN or non-string

    indep_X_encoded['Duration'] = indep_X_encoded['Duration'].apply(parse_duration)

    # Define models
    lin = LogisticRegression(max_iter=1000, random_state=0)
    dec = DecisionTreeClassifier(random_state=0)
    rf = RandomForestClassifier(n_estimators=10, random_state=0)
    xgb = XGBClassifier(n_jobs=5, learning_rate=0.1, max_depth=10, random_state=1)

    rfemodellist = [lin, dec, rf, xgb]

    for model in rfemodellist:
        # Recursive Feature Elimination
        log_rfe = RFE(estimator=model, n_features_to_select=n)
        log_fit = log_rfe.fit(indep_X_encoded, dep_Y)  # Use encoded data
        log_rfe_feature = log_fit.transform(indep_X_encoded)
        rfelist.append(log_rfe_feature)

        # Selected feature names
        selected_columns = [col for col, selected in zip(indep_X_encoded.columns, log_rfe.support_) if selected]
        colnames_list.append(selected_columns)

        # Train/test split (replace split_scalar with sklearn train_test_split)
        X_train, X_test, y_train, y_test = train_test_split(
            log_rfe_feature, dep_Y, test_size=0.2, random_state=42, stratify=dep_Y
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Scores
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average="weighted")
        scores_list.append((acc, f1))   # store tuple

    return rfelist, colnames_list, scores_list


# Example usage
rfelist, colnames_list, scores_list = rfeFeature(X, y, 5)

# Print results
for model_name, selected_columns, scores in zip(["Logistic", "Decision", "Random", "XGBoost"], colnames_list, scores_list):
    acc, f1 = scores
    print(f"Model: {model_name}")
    print("Selected Columns:", selected_columns)
    print(f"Accuracy: {acc:.3f}, F1 Score: {f1:.3f}\n")"""


'from sklearn.feature_selection import RFE\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.tree import DecisionTreeClassifier\nfrom sklearn.ensemble import RandomForestClassifier\nfrom xgboost import XGBClassifier\nfrom sklearn.metrics import accuracy_score, f1_score\nfrom sklearn.model_selection import train_test_split\nimport pandas as pd\n\ndef rfeFeature(indep_X, dep_Y, n):\n    rfelist = []\n    colnames_list = []\n    scores_list = []  # will store (accuracy, f1) for each model\n\n    # Convert categorical features to numerical using one-hot encoding\n    indep_X_encoded = pd.get_dummies(indep_X, columns=[\'Airline\', \'Source\', \'Destination\'])\n\n    # Convert \'Duration\' to numerical (assuming format \'Xh Ym\')\n    def parse_duration(duration_str):\n        if isinstance(duration_str, str):\n            parts = duration_str.split(\' \')\n            hours = 0\n            minutes = 0\n            for part in parts:\n                if \'h\' in part:\n   

In [26]:
"""from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import pandas as pd

def rfeFeature(indep_X, dep_Y, n):
    rfelist = []
    colnames_list = []
    scores_list = []  # will store (accuracy, f1) for each model

    # Convert categorical features to numerical using one-hot encoding
    indep_X_encoded = pd.get_dummies(indep_X, columns=['Airline', 'Source', 'Destination'])

    # Convert 'Duration' to numerical (assuming format 'Xh Ym')
    def parse_duration(duration_str):
        if isinstance(duration_str, str):
            parts = duration_str.split(' ')
            hours = 0
            minutes = 0
            for part in parts:
                if 'h' in part:
                    hours = int(part.replace('h', ''))
                elif 'm' in part:
                    minutes = int(part.replace('m', ''))
            return hours * 60 + minutes
        return 0  # Handle NaN or non-string

    indep_X_encoded['Duration'] = indep_X_encoded['Duration'].apply(parse_duration)

    # Define models
    lin = LogisticRegression(max_iter=1000, random_state=0)
    dec = DecisionTreeClassifier(random_state=0)
    rf = RandomForestClassifier(n_estimators=10, random_state=0)
    xgb = XGBClassifier(n_jobs=5, learning_rate=0.1, max_depth=10, random_state=1)

    rfemodellist = [lin, dec, rf, xgb]

    for model in rfemodellist:
        # Recursive Feature Elimination
        log_rfe = RFE(estimator=model, n_features_to_select=n)
        log_fit = log_rfe.fit(indep_X_encoded, dep_Y)  # Use encoded data
        log_rfe_feature = log_fit.transform(indep_X_encoded)
        rfelist.append(log_rfe_feature)

        # Selected feature names
        selected_columns = [col for col, selected in zip(indep_X_encoded.columns, log_rfe.support_) if selected]
        colnames_list.append(selected_columns)

        # Train/test split (replace split_scalar with sklearn train_test_split)
        X_train, X_test, y_train, y_test = train_test_split(
            log_rfe_feature, dep_Y, test_size=0.2, random_state=42, stratify=dep_Y
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Scores
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average="weighted")
        scores_list.append((acc, f1))   # store tuple

    return rfelist, colnames_list, scores_list"""




'from sklearn.feature_selection import RFE\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.tree import DecisionTreeClassifier\nfrom sklearn.ensemble import RandomForestClassifier\nfrom xgboost import XGBClassifier\nfrom sklearn.metrics import accuracy_score, f1_score\nfrom sklearn.model_selection import train_test_split\nimport pandas as pd\n\ndef rfeFeature(indep_X, dep_Y, n):\n    rfelist = []\n    colnames_list = []\n    scores_list = []  # will store (accuracy, f1) for each model\n\n    # Convert categorical features to numerical using one-hot encoding\n    indep_X_encoded = pd.get_dummies(indep_X, columns=[\'Airline\', \'Source\', \'Destination\'])\n\n    # Convert \'Duration\' to numerical (assuming format \'Xh Ym\')\n    def parse_duration(duration_str):\n        if isinstance(duration_str, str):\n            parts = duration_str.split(\' \')\n            hours = 0\n            minutes = 0\n            for part in parts:\n                if \'h\' in part:\n   

In [27]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split


def optimizedFeatureSelection(indep_X, dep_Y, n=5, test_size=0.2, random_state=42):
    colnames_list = []
    scores_list = []

    X_processed = indep_X.copy()

    # One-hot encode categorical columns safely
    for col in ['Airline', 'Source', 'Destination']:
        if col in X_processed.columns:
            X_processed = pd.get_dummies(X_processed, columns=[col], drop_first=True)

    # Convert 'Duration' column to minutes
    if 'Duration' in X_processed.columns:
        def parse_duration(x):
            if isinstance(x, str):
                parts = x.split(' ')
                hours, minutes = 0, 0
                for part in parts:
                    if 'h' in part:
                        hours = int(part.replace('h', ''))
                    elif 'm' in part:
                        minutes = int(part.replace('m', ''))
                return hours * 60 + minutes
            try:
                return int(float(x))  # numeric values
            except:
                return 0
        X_processed['Duration'] = X_processed['Duration'].map(parse_duration)

    # Drop classes with less than 2 samples (avoids stratify error)
    class_counts = dep_Y.value_counts()
    valid_classes = class_counts[class_counts > 1].index
    mask = dep_Y.isin(valid_classes)
    X_processed = X_processed.loc[mask]
    dep_Y = dep_Y.loc[mask]

    # Feature selection (mutual info works for classification)
    selector = SelectKBest(score_func=mutual_info_classif, k=min(n, X_processed.shape[1]))
    X_new = selector.fit_transform(X_processed, dep_Y)

    # ✅ Flatten mask before selecting columns
    support_mask = selector.get_support().ravel()
    selected_columns = X_processed.columns[support_mask]
    colnames_list = list(selected_columns)

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_new, dep_Y, test_size=test_size, random_state=random_state, stratify=dep_Y
    )

    # Classification models
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=0),
        "Decision Tree": DecisionTreeClassifier(random_state=0),
        "Random Forest": RandomForestClassifier(n_estimators=50, random_state=0),
        "XGBoost": XGBClassifier(
            n_jobs=5, learning_rate=0.1, max_depth=10,
            random_state=1, use_label_encoder=False, eval_metric="mlogloss"
        )
    }

    # Train & evaluate
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average="weighted")
        scores_list.append((name, acc, f1))

    return colnames_list, scores_list


# Example call
selected_columns, scores_list = optimizedFeatureSelection(X, y, n=5)

print("✅ Selected Features:", selected_columns, "\n")
print("📊 Model Performance:\n")
for model_name, acc, f1 in scores_list:
    print(f"Model: {model_name}")
    print(f"  Accuracy : {acc:.3f}")
    print(f"  F1 Score : {f1:.3f}\n")


ValueError: Cannot index with multidimensional key